In [1]:
#import the relevant functions
from modify_acudata import *
from datetime import datetime

## 1. Running the time shifting operation (for asynchronous acu sim. and detector data streams)

In [1]:
#define the relevant variables

#list of acu g3 files to be modified
flist = ['/mnt/home/ssarkar1/work/workspace/pcam_data/17533/1753319214.g3',
         '/mnt/home/ssarkar1/work/workspace/pcam_data/17533/1753322841.g3']
#location where the modified g3 files will be written
outloc = '/mnt/home/ssarkar1/work/workspace/bookbinder/mock_l2/hk/test2'

#original start time of the acu simulated data streams
otime = 175331923236886368.0 #10ns
#original start time of the detector data streams, acu times will be shifted to
#this time window
mtime = 174976193376301400.0 #10ns


In [2]:
run_timeshift(flist, outloc, otime, mtime)

NameError: name 'run_timeshift' is not defined

In [ ]:
#compare the original times and inspect the modified files

print ("Original file time info:")
inspect_file(flist[0])


test_f = outloc+'/1753319214_m.g3'
print ("Modified file time info:")
acu_starttime = inspect_file(test_f)


In [ ]:
detf = '/mnt/home/ssarkar1/work/workspace/bookbinder/mock_l2/rfsoc/17497/rfsoc5_drone1/r05d1_1749761933_000.g3'
g3f = g3.G3File(detf)

frame = g3f.next()
print ("Detector start time: ", frame['time'])
print ("Shifted ACU data start time: ", datetime.fromtimestamp(acu_starttime))

## 2. Bookbinder trial run from given list of files

In [ ]:
import sys
sys.path.append('/mnt/work/ssarkar1/sotodlib/sotodlib/io/')
from bookbinder import *

In [ ]:
%load_ext memory_profiler
%load_ext line_profiler

In [ ]:
#define the input arguments
rdir = '/mnt/work/ssarkar1/workspace/bookbinder/mock_l2'
odir = '/mnt/home/ssarkar1/work/workspace/bookbinder/mock_books/17533'

hkloc = rdir+'/hk/test2/'
hkfiles = [hkloc+'1753319214_m.g3',
          hkloc+'1753322841_m.g3']

detloc = rdir+'/rfsoc/17497/rfsoc5_drone1/'
detfiles = {'r05d1_1749761933': [detloc+'r05d1_1749761933_000.g3',
                                detloc+'r05d1_1749761933_001.g3',
                                detloc+'r05d1_1749761933_002.g3',
                                detloc+'r05d1_1749761933_003.g3',
                                detloc+'r05d1_1749761933_004.g3',
                                detloc+'r05d1_1749761933_005.g3']}
hkfield = {'az' : 'observatory.acu1.feeds.Azimuth',
           'el' : 'observatory.acu1.feeds.Elevation'}

bb = BookBinder(rdir, odir, hkfield, hkfiles, detfiles, allow_bad_timing=True)
s

In [ ]:
#run bookbinding
%%memit
bb.bind(pbar=True)

## 3. Inspect Book

In [ ]:
from glob import glob

In [ ]:
bookloc = "/mnt/home/ssarkar1/work/workspace/bookbinder/mock_books/17533/"
bookfiles = sorted(glob(bookloc+"D_*"))

In [ ]:
def get_streams(files):
    
    azarr = []
    elarr = []
    Iarr = []
    Qarr = []
    tarr = []
    for file in files:
        print(f"Processing {file}")
        g3f = g3.G3File(file)
        while True:
            try:
                frame = g3f.next()
            except:
                break
            if frame.type == g3.G3FrameType.Scan:
                ancil = frame['ancil']
                signal = frame['signal']
                azarr.append(np.array(ancil['az_enc']))
                elarr.append(np.array(ancil['el_enc']))
                Iarr.append(signal.data[::2][0,:])#only extracting one channel for the purpose of inspection
                Qarr.append(signal.data[1::2][0,:])
                tarr.append(np.array(signal.times)/g3.G3Units.s)#unix
    return np.hstack(azarr), np.hstack(elarr), np.hstack(Iarr), np.hstack(Qarr), np.hstack(tarr)

In [ ]:
AZ, EL, dI, dQ, T = get_streams(bookfiles[:2])#using only two files, full book is too large to be handled in single arrays

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rc('figure', dpi=300)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 6))

ax1.plot(T-T[0], dI, label='I Channel')
ax1.plot(T-T[0], dQ, label='Q Channel')
ax1.legend()
ax1.grid(True)
ax1.set_ylabel('Signal')

ax2.plot(T-T[0], AZ, label='Azimuth')
ax2.plot(T-T[0], EL, label='Elevation')
ax2.legend()
ax2.grid(True)
ax2.set_ylabel('Ancil')

ax2.set_xlabel('Elapsed Time (s)')

plt.tight_layout()
plt.show()

## New Code

In [2]:
import spt3g.core as core

#import the relevant functions
from modify_acudata import *
from datetime import datetime

det_file = '/data/shwetha/det_files/rfsoc01_drone1/r01d1_1755824003_000.g3'
acu_file = '/data/shwetha/hk_files/17653/1765305841.g3'

# --- ACU ---
g3f = g3.G3File(acu_file)
frame = g3f.next()
print("ACU start time :", frame['start_time'])


# --- DETECTOR ---
g3f = g3.G3File(det_file)
frame = g3f.next()
det_time = frame['time']
# Convert G3Time to UNIX time
det_unix_time = det_time.time / 1e8

print("Detector start time :", det_unix_time)


ACU start time : 1764101314.8922803
Detector start time : 1755824003.772942


In [3]:
# ist of acu g3 files to be modified
flist = ['/data/shwetha/hk_files/17653/1765305841.g3']

# location where the modified g3 files will be written
outloc = '/data/shwetha/hk_files/timeshifted'

# original start time of the acu simulated data streams
otime = 1764101314.8922803

# original start time of the detector data streams, acu times will be shifted to
# this time window
mtime = 1755824003.772942

run_timeshift(flist, outloc, otime, mtime)

offset = mtime - otime
print("Time shift offset:", offset)

Processing file: /data/shwetha/hk_files/17653/1765305841.g3
Reached the end of the file
Done.
Time shift offset: -8277311.119338274


In [4]:
import spt3g.core as core
#import spt3g.core.g3reader as g3

det_file = '/data/shwetha/det_files/rfsoc01_drone1/r01d1_1755824003_000.g3'
acu_file = '/data/shwetha/hk_files/timeshifted/1765305841_m.g3'

# --- ACU ---
g3f = g3.G3File(acu_file)
frame = g3f.next()
print("ACU start time :", frame['start_time'])

# --- DETECTOR ---
g3f = g3.G3File(det_file)
frame = g3f.next()
det_time = frame['time']

# Convert G3Time → UNIX time
det_unix_time = det_time.time / 1e8

print("Detector start time :", det_unix_time)


ACU start time : 1755824003.772942
Detector start time : 1755824003.772942


In [ ]:
1764101314.8922803